In [1]:
from pathlib import Path

from tapas_gmm.env.rlbench import RLBenchEnvironment, RLBenchEnvironmentConfig
from tapas_gmm.policy.gmm import GMMPolicy, GMMPolicyConfig
from tapas_gmm.policy.models.tpgmm import AutoTPGMMConfig

from rlbench.action_modes.arm_action_modes import BimanualEndEffectorPoseViaIK

import imageio.v2 as imageio


2026-07-08 16:34:21.584 | INFO     |  Running on cpu


/home/nils/Documents/Study Project/Code/riepybdlib/riepybdlib/data.py:34: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_listdir


In [2]:
model_path = Path(
    "../outputs/bimanual_tpgmm.pkl"
)

In [3]:
tapas_env = RLBenchEnvironment(
    RLBenchEnvironmentConfig(
        action_mode = BimanualEndEffectorPoseViaIK,
        robot_setup = "dual_panda",
        task = "BimanualDualPushButtons",
        cameras = ("front",),
        camera_pose = {},
        image_size = (128, 128),
        static = False,
        headless = False,
        scale_action = False,
        delay_gripper = False,
        gripper_plot = False,   
    )
)

policy = GMMPolicy(
    GMMPolicyConfig(
        suffix=None,
        model = AutoTPGMMConfig(),
        batch_predict_in_t_models = False,
        topp_in_t_models = False,
        binary_gripper_action = True,
        force_overwrite_checkpoint_config = True,
        pos_lag_thresh=0.03,
    )
)

2026-07-08 16:34:28.849 | INFO     |  Initializing Policy:
2026-07-08 16:34:28.850 | INFO     |  No encoder config provided. Using None.
None


In [4]:
obs = tapas_env.reset()
policy.from_disk(str(model_path))
policy.eval()
policy.reset_episode(tapas_env)

2026-07-08 16:34:30.336 | INFO     |  Loading model:
2026-07-08 16:34:30.434 | ERROR    |  Config mismatch
root.demos_segmentation.distance_based False != True
root.demos_segmentation.components_prop_to_len True != False
root.demos_segmentation.velocity_based True != False
root.tpgmm.reg_em_finish_diag 0.0002 != 0.001
root.tpgmm.add_action_component False != True
root.tpgmm.add_time_component True != False
root.tpgmm.reg_em_finish_diag_gripper 0.02 != 0.1
root.tpgmm.add_gripper_action True != False
root.tpgmm.reg_diag  0.0002 != 0.001
root.tpgmm.reg_init_diag 0.0005 != 5e-05
root.tpgmm.reg_diag_gripper 0.02 != 0.1

2026-07-08 16:34:30.435 | WARNING  |  Overwriting config. This can lead to unexpected errors.
2026-07-08 16:34:30.435 | INFO     |  Detected time-based model: True. Using time-driven policy. Set time_based in config to overwrite.
2026-07-08 16:34:30.435 | INFO     |  Creating local marginals
2026-07-08 16:34:30.435 | INFO     |  Changing number of components to 3


In [5]:
total_reward = 0
frames = []

for step in range(450):
    action, info = policy.predict(obs)
    obs, reward, done, env_info = tapas_env.step(action)
    frame = obs.cameras["front"].rgb

    if frame.ndim == 4:
        frame = frame[0]

    if frame.shape[0] == 3:
        frame = frame.permute(1, 2, 0)

    frame = (frame * 255).clip(0, 255).byte().cpu().numpy()
    frames.append(frame)
    total_reward += reward

    if obs is None or done or info.get("done", False):
        break

imageio.mimsave("../outputs/bimanual_tpgmm_run.mp4", frames, fps=20)

print("steps:", step + 1)
print("total_reward:", total_reward)

tapas_env.close()

2026-07-08 16:34:30.454 | WARNING  |  Implementation lacking modulo rots, enforce z-up/down, etc.
2026-07-08 16:34:30.534 | INFO     |  Action [-1.67712899e-05 -5.84121468e-06  2.67060201e-04  0.00000000e+00
  0.00000000e+00  0.00000000e+00  1.00000000e+00  1.00000000e+00
  0.00000000e+00  1.14431339e-03 -3.40515267e-04 -2.21330310e-03
  0.00000000e+00  3.46944695e-18 -3.46944695e-18  1.00000000e+00
  1.00000000e+00  0.00000000e+00]


/home/nils/Documents/Study Project/Code/TAPAS/tapas_gmm/utils/geometry_np.py:84: RuntimeWarning: invalid value encountered in arccos
  theta = 2 * np.arccos(2 * np.dot(q, r) ** 2 - 1)


2026-07-08 16:34:30.781 | INFO     |  Pos lag: [[ 7.63433647e-04  2.05349596e-04 -1.08122966e-03]
 [-1.15232923e-04 -7.58747907e-06 -2.48305452e-04]], quat lag: nan, pos change [[ 8.4149837e-04  2.7684867e-04 -7.9381466e-04]
 [ 1.9559264e-04  1.7434359e-06  5.0354004e-04]], quat change 0.0013810679388648144
2026-07-08 16:34:30.782 | INFO     |  Action [ 5.20941548e-05 -7.46864105e-06 -2.70270009e-04  4.23856825e-06
 -3.97566507e-04  1.01917165e-05  9.99999921e-01  1.00000000e+00
  0.00000000e+00  4.96542131e-04 -1.39444913e-04 -1.23976189e-03
 -2.99014408e-06 -4.09917054e-04  7.22066587e-07  9.99999916e-01
  1.00000000e+00  0.00000000e+00]
2026-07-08 16:34:30.975 | INFO     |  Pos lag: [[ 4.46217901e-04  7.34409868e-05 -7.44565561e-04]
 [-2.54354272e-04  1.62399540e-05 -5.39884095e-04]], quat lag: 0.0021858752919391186, pos change [[ 3.1664968e-04  1.3325363e-04 -3.4093857e-04]
 [ 1.3913214e-04 -2.3707747e-05  2.8991699e-04]], quat change 0.0
2026-07-08 16:34:30.976 | INFO     |  Actio

2026-07-08 16:34:31.590 | INFO     |  Pos lag: [[ 2.89102335e-04  7.57364948e-05 -7.74864285e-04]
 [-2.69259016e-04 -2.66453294e-06 -5.74329817e-04]], quat lag: 0.00431541753602887, pos change [[-8.41319561e-05  6.02677464e-05 -1.16825104e-04]
 [-1.38282776e-05 -1.65849924e-05 -6.19888306e-06]], quat change 0.00569428944063524
2026-07-08 16:34:31.592 | INFO     |  Action [ 1.28218717e-03 -3.85610337e-04  4.19175746e-03  9.91266796e-03
 -4.64345273e-04  8.15103133e-02  9.96623094e-01  1.00000000e+00
  0.00000000e+00 -8.91091057e-04 -1.08912273e-03 -2.86946356e-03
  7.20036975e-04  1.64305804e-02  6.02464627e-03  9.99846599e-01
  1.00000000e+00  0.00000000e+00]


2026-07-08 16:34:31.993 | INFO     |  Pos lag: [[-2.90818135e-04  1.04360120e-03 -2.85292209e-03]
 [-9.49019772e-05  1.60787364e-04 -6.58862561e-04]], quat lag: 0.06923944284877205, pos change [[ 1.2189150e-05  2.5369227e-05 -1.4603138e-04]
 [-1.5108287e-04 -5.4399669e-04  5.0356388e-03]], quat change 0.3174465809452404
2026-07-08 16:34:31.995 | INFO     |  Action [ 1.90597975e-03 -6.74908015e-04  7.89057518e-03  1.81242374e-02
 -2.19599791e-04  1.51183801e-01  9.88339477e-01  1.00000000e+00
  0.00000000e+00 -2.88976527e-03 -3.16901021e-03 -7.09256904e-03
  2.02897955e-03  4.75462173e-02  1.69410945e-02  9.98723305e-01
  1.00000000e+00  0.00000000e+00]


...

2026-07-08 16:34:34.432 | INFO     |  Pos lag: [[-1.70531040e-03  3.58157248e-03 -8.80887558e-03]
 [-2.61751951e-04 -2.06452544e-06 -5.87866801e-04]], quat lag: 0.23795733300794686, pos change [[ 1.8566847e-05 -1.1697412e-06 -3.9696693e-05]
 [ 1.8745661e-05 -3.1650066e-05 -1.3113022e-06]], quat change nan
2026-07-08 16:34:34.434 | INFO     |  Action [ 8.59641423e-05 -1.00955198e-04 -6.29700783e-04 -3.53406603e-04
 -5.51095750e-04  5.83691359e-05  9.99999784e-01  1.00000000e+00
  0.00000000e+00 -3.43654408e-03 -3.71583327e-03 -8.22890054e-03
  2.38437444e-03  5.59818985e-02  1.98844508e-02  9.98230910e-01
  1.00000000e+00  0.00000000e+00]
2026-07-08 16:34:34.571 | INFO     |  Pos lag: [[-1.68182626e-03  3.58438895e-03 -8.83307543e-03]
 [-3.07945547e-04 -1.34639660e-05 -6.69286067e-04]], quat lag: 0.2378101095123585, pos change [[-2.3484230e-05 -2.8163195e-06  2.4199486e-05]
 [ 4.6193600e-05  1.1399388e-05  8.1419945e-05]], quat change nan
2026-07-08 16:34:34.572 | INFO     |  Action [ 1

2026-07-08 16:34:34.723 | INFO     |  Pos lag: [[-1.65273920e-03  3.58634102e-03 -8.86764615e-03]
 [-3.09122739e-04 -6.25180813e-06 -7.00518847e-04]], quat lag: 0.23775911386334447, pos change [[-2.9087067e-05 -1.9520521e-06  3.4570694e-05]
 [ 1.1771917e-06 -7.2121620e-06  3.1232834e-05]], quat change nan
2026-07-08 16:34:34.724 | INFO     |  Action [ 9.82898834e-05 -1.22196633e-04 -7.49486308e-04 -4.20328742e-04
 -6.32297159e-04  5.06799471e-05  9.99999710e-01  1.00000000e+00
  0.00000000e+00 -3.39830932e-03 -3.71600855e-03 -8.29887400e-03
  2.38429117e-03  5.59292078e-02  1.98856144e-02  9.98233841e-01
  1.00000000e+00  0.00000000e+00]
2026-07-08 16:34:34.829 | INFO     |  Pos lag: [[-1.70602575e-03  3.57728111e-03 -8.80780309e-03]
 [-2.49458489e-04 -1.46858657e-05 -6.53192754e-04]], quat lag: 0.23793125276029248, pos change [[ 5.3286552e-05  9.0599060e-06 -5.9843063e-05]
 [-5.9664249e-05  8.4340572e-06 -4.7326088e-05]], quat change nan
2026-07-08 16:34:34.830 | INFO     |  Action [ 

2026-07-08 16:34:34.983 | INFO     |  Pos lag: [[-1.69583336e-03  3.57813048e-03 -8.83176416e-03]
 [-2.22353277e-04 -1.55650342e-05 -5.97402806e-04]], quat lag: 0.2378904288500473, pos change [[-1.0192394e-05 -8.4936619e-07  2.3961067e-05]
 [-2.7105212e-05  8.7916851e-07 -5.5789948e-05]], quat change nan
2026-07-08 16:34:34.985 | INFO     |  Action [ 4.43137300e-05 -9.15220432e-05 -6.29470713e-04 -3.63812827e-04
 -5.66385572e-04  4.52512059e-05  9.99999772e-01  1.00000000e+00
  0.00000000e+00 -3.43284588e-03 -3.71156388e-03 -8.25341172e-03
  2.38352032e-03  5.59653611e-02  1.98801345e-02  9.98231926e-01
  1.00000000e+00  0.00000000e+00]
2026-07-08 16:34:35.134 | INFO     |  Pos lag: [[-1.72837750e-03  3.57605922e-03 -8.76977533e-03]
 [-1.39279303e-04 -1.51323924e-06 -4.45530172e-04]], quat lag: 0.2380201071081131, pos change [[ 3.2544136e-05  2.0712614e-06 -6.1988831e-05]
 [-8.3073974e-05 -1.4051795e-05 -1.5187263e-04]], quat change 0.001953125019402554
2026-07-08 16:34:35.135 | INFO  

2026-07-08 16:34:43.631 | INFO     |  Pos lag: [[-0.0002889  -0.00023328 -0.00026901]
 [ 0.00038326  0.00028498 -0.00036894]], quat lag: 0.014271447134872649, pos change [[ 0.00470404 -0.01179205  0.0406372 ]
 [-0.00050035 -0.00188711  0.0227859 ]], quat change 0.893190235904872
2026-07-08 16:34:43.632 | INFO     |  Action [ 4.67565211e-03 -4.55615928e-03  3.15213891e-02  1.09750791e-01
 -6.33584727e-04  9.05982146e-01  4.08840694e-01  1.00000000e+00
  0.00000000e+00  8.61952941e-03  1.18800881e-02  4.80709253e-02
  1.30032925e-03  3.78613933e-02  1.38005880e-02  9.99186853e-01
  0.00000000e+00  0.00000000e+00]
2026-07-08 16:34:44.577 | INFO     |  Pos lag: [[-0.00032035 -0.00016746 -0.0005317 ]
 [-0.00054969 -0.00188611  0.03204881]], quat lag: 4.5862903195790095, pos change [[ 4.1260719e-03 -1.1673883e-02  4.9230218e-02]
 [ 1.0728836e-06  3.1945109e-04  9.8705292e-05]], quat change 0.16353823115891558
2026-07-08 16:34:44.578 | INFO     |  Action [ 5.55139877e-03 -5.93330684e-03  3.84

2026-07-08 16:34:53.133 | INFO     |  Pos lag: [[-2.08156490e-04 -1.89046713e-04 -2.32590073e-04]
 [ 4.87848762e-04  3.78958442e-05 -6.57041901e-04]], quat lag: 0.0029515059037257653, pos change [[-4.9404800e-05 -2.1398067e-05 -8.7141991e-05]
 [-4.9293041e-05 -5.0067902e-06  7.3730946e-05]], quat change 0.0
2026-07-08 16:34:53.137 | INFO     |  Action [ 4.68700892e-04 -1.86272847e-04  2.29890443e-03  1.21552010e-01
  2.65785685e-04  9.92584037e-01 -1.40293528e-03  0.00000000e+00
  0.00000000e+00 -1.27809989e-04  2.51344683e-04 -2.37604113e-04
 -2.89763139e-04 -2.55089198e-04 -7.42767178e-05  9.99999923e-01
  0.00000000e+00  0.00000000e+00]
2026-07-08 16:34:53.280 | INFO     |  Pos lag: [[-0.0001961  -0.00022345 -0.00028471]
 [ 0.00047427  0.00017397  0.00227648]], quat lag: 6.277583348949003, pos change [[ 6.7897141e-05 -2.8133392e-05  4.7564507e-05]
 [-8.8512897e-06  8.8512897e-06  2.3365021e-05]], quat change nan
2026-07-08 16:34:53.280 | INFO     |  Action [ 6.45581905e-04 -3.379181

In [6]:
tapas_env.close()